In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
import datetime
from sklearn import metrics
import shap
from pygam import LinearGAM, s
from matplotlib import rcParams
from matplotlib.ticker import MaxNLocator
import warnings
warnings.filterwarnings('ignore')

# ======================
# User Input Configuration
# ======================
INPUT_CSV_FILE = "D:\\LP_LST_Data\\Beijing_Annual.csv"

# ======================
# Global Settings
# ======================
def setup_plot_style():
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['ps.fonttype'] = 42
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['axes.unicode_minus'] = False

# ======================
# Global Variables
# ======================
cities_r2_data = []           # Store R² metrics for cities
cities_best_params = []       # Store best model parameters
cities_complete_features = [] # Store complete feature importance data

# ======================
# Feature Column Definitions
# ======================
SPECIFIED_FEATURES = [
    'PGS_PLAND', 'PGS_LSI', 'PGS_ENN_AM', 'PGS_ED','WBS_PLAND', 'WBS_LSI', 'WBS_ENN_AM', 'WBS_ED'
]

# ======================
# Core Processing Function
# ======================
def process_single_city_complete(csv_path, city_name, output_dir):
    try:
        os.makedirs(output_dir, exist_ok=True)
        original_cwd = os.getcwd()
        os.chdir(output_dir)
        print(f"Output directory: {output_dir}")
        
        setup_plot_style()
        
        # Data loading
        print("Reading data...")
        df = pd.read_csv(csv_path)

        # Specify features and target
        X = df[SPECIFIED_FEATURES]
        y = df['LST']
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )
        
        print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
        
        # Model training with grid search
        print("Training and optimizing model...")
        
        params_xgb = {
            'learning_rate': 0.01,
            'booster': 'gbtree',
            'objective': 'reg:squarederror',
            'max_leaves': 137,
            'verbosity': 1,
            'seed': 42,
            'nthread': -1,
            'colsample_bytree': 0.6,
            'subsample': 0.7,
            'eval_metric': 'rmse'
        }
        
        model_xgb = xgb.XGBRegressor(**params_xgb)
        
        param_grid = {
            'n_estimators': [100, 200, 300, 400, 500],
            'max_depth': [3, 4, 5, 6, 7],
            'learning_rate': [0.01, 0.02, 0.05, 0.1],
        }
        
        grid_search = GridSearchCV(
            estimator=model_xgb,
            param_grid=param_grid,
            scoring='neg_mean_squared_error',
            cv=10,
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        
        print(f"Best parameters: {grid_search.best_params_}")
        best_rmse = (-grid_search.best_score_) ** 0.5
        print(f"Best RMSE: {best_rmse:.4f}")
        
        # Model evaluation
        print("Evaluating model...")
        
        y_pred_train = best_model.predict(X_train)
        y_pred_test = best_model.predict(X_test)
        
        mse_train = metrics.mean_squared_error(y_train, y_pred_train)
        rmse_train = np.sqrt(mse_train)
        mae_train = metrics.mean_absolute_error(y_train, y_pred_train)
        r2_train = metrics.r2_score(y_train, y_pred_train)
        
        mse_test = metrics.mean_squared_error(y_test, y_pred_test)
        rmse_test = np.sqrt(mse_test)
        mae_test = metrics.mean_absolute_error(y_test, y_pred_test)
        r2_test = metrics.r2_score(y_test, y_pred_test)
        
        print("\nTraining set metrics:")
        print(f"MSE: {mse_train:.4f}, RMSE: {rmse_train:.4f}, MAE: {mae_train:.4f}, R²: {r2_train:.4f}")
        print("\nTest set metrics:")
        print(f"MSE: {mse_test:.4f}, RMSE: {rmse_test:.4f}, MAE: {mae_test:.4f}, R²: {r2_test:.4f}")
        
        # SHAP analysis
        print("Calculating SHAP values...")
        
        explainer = shap.TreeExplainer(best_model)
        shap_values_numpy = explainer.shap_values(X_test)
        shap_values_Explanation = explainer(X_test)
        
        print("Calculating SHAP interaction values...")
        shap_interaction_values = explainer.shap_interaction_values(X_test)
        print(f"SHAP interaction values shape: {shap_interaction_values.shape}")
        
        # Calculate feature importance
        mean_abs_shap = np.mean(np.abs(shap_values_Explanation.values), axis=0)
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': mean_abs_shap
        }).sort_values('importance', ascending=False)
        
        # Collect statistics for summary tables
        global all_cities_r2_data
        all_cities_r2_data.append({
            'City': city_name,
            'Train_R2': r2_train,
            'Test_R2': r2_test,
            'Train_RMSE': rmse_train,
            'Test_RMSE': rmse_test,
            'Sample_Size': len(df),
            'Num_Features': X.shape[1]
        })
        
        global all_cities_best_params
        best_params_dict = grid_search.best_params_.copy()
        best_params_dict['City'] = city_name
        best_params_dict['Best_RMSE'] = best_rmse
        all_cities_best_params.append(best_params_dict)
        
        global all_cities_complete_features
        total_importance = feature_importance['importance'].sum()
        if total_importance > 0:
            feature_importance['relative_importance'] = feature_importance['importance'] / total_importance
        else:
            feature_importance['relative_importance'] = 0
        
        complete_feature_dict = {'City': city_name}
        for feature in SPECIFIED_FEATURES:
            if feature in feature_importance['feature'].values:
                importance_value = feature_importance[feature_importance['feature'] == feature]['relative_importance'].values[0]
                complete_feature_dict[feature] = importance_value
            else:
                complete_feature_dict[feature] = 0
        all_cities_complete_features.append(complete_feature_dict)
        
        # Generate visualizations
        print("Generating visualizations...")
        
        # 1. SHAP beeswarm plot (global feature importance)
        create_shap_beeswarm_plot(shap_values_Explanation, X_test, X.columns, city_name)
        
        # 2. GAM main effects plots (individual feature effects)
        create_gam_plots_complete(shap_values_Explanation, X_test, city_name)
        
        # 3. SHAP interaction matrix (feature interactions)
        print("Generating interaction plots (8x8 matrix)...")
        create_interaction_plots_original(shap_interaction_values, X_test, city_name)
        
        # Export summary tables
        generate_r2_summary_table(output_dir, city_name)
        generate_best_params_summary_table(output_dir, city_name)
        generate_complete_features_summary_table(output_dir, city_name)
        
        os.chdir(original_cwd)
        
        print(f"\n✓ {city_name} processing completed. All results saved to: {output_dir}")
        
        return {
            'City': city_name,
            'Sample_Size': len(df),
            'Num_Features': X.shape[1],
            'Best_RMSE': best_rmse,
            'Test_R2': r2_test,
            'Train_R2': r2_train,
            'Status': 'Success',
            'Output_Dir': output_dir
        }
        
    except Exception as e:
        print(f"Error processing {city_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        if 'original_cwd' in locals():
            os.chdir(original_cwd)
        return {
            'City': city_name,
            'Sample_Size': 0,
            'Num_Features': 0,
            'Best_RMSE': 0,
            'Test_R2': 0,
            'Train_R2': 0,
            'Status': f'Failed: {str(e)}',
            'Output_Dir': ''
        }

# ======================
# SHAP Visualization Functions
# ======================

def create_shap_beeswarm_plot(shap_values_Explanation, X_test, feature_names, city_name):
    try:
        features = feature_names
        mean_shap_value1 = abs(shap_values_Explanation.values).mean(0)
        
        shap_summary = pd.DataFrame({
            "Feature": features,
            "MeanSHAP": mean_shap_value1
        }).sort_values("MeanSHAP", ascending=False)
        
        sorted_feats = shap_summary["Feature"].values
        feat_idx = [list(features).index(f) for f in sorted_feats]
        
        X_test_sorted = X_test[sorted_feats]
        shap_vals = shap_values_Explanation.values
        shap_vals_sorted = shap_vals[:, feat_idx]
        
        mean_shap = shap_summary["MeanSHAP"].values
        mean_shap_pct = mean_shap / mean_shap.sum() * 100
        
        fig = plt.figure(figsize=(6, 5), dpi=150)
        ax_bee = fig.add_axes([0.3, 0.1, 0.6, 0.8])
        
        cmap_name = "RdYlBu"
        try:
            cmap = plt.colormaps[cmap_name]
            cmap1 = plt.colormaps[cmap_name]
        except AttributeError:
            cmap = plt.cm.get_cmap(cmap_name)
            cmap1 = plt.cm.get_cmap(cmap_name)
        
        shap.plots._utils.colors.red_blue = cmap
        
        explainer_data = shap.Explanation(
            values=shap_vals_sorted,
            data=X_test_sorted.values,
            feature_names=sorted_feats
        )
        
        current_ax = plt.gca()
        plt.sca(ax_bee)
        shap.plots.beeswarm(explainer_data, show=False, plot_size=None)
        plt.sca(current_ax)
        
        y_coords = ax_bee.get_yticks()
        y_labels = [tick.get_text() for tick in ax_bee.get_yticklabels()]
        y_order = y_labels if any(lbl != "" for lbl in y_labels) else sorted_feats
        
        shap_mean_map = dict(zip(sorted_feats, mean_shap))
        shap_pct_map = dict(zip(sorted_feats, mean_shap_pct))
        bar_widths = [shap_mean_map.get(lbl, 0) for lbl in y_order]
        bar_pcts1 = [shap_pct_map.get(lbl, 0) for lbl in y_order]
        bar_pcts = bar_pcts1.copy()
        if len(bar_pcts) > 0:
            bar_pcts[0] = 100 - np.sum(bar_pcts[1:])
        
        ax_bar = ax_bee.twiny()
        ax_bar.set_zorder(0)
        ax_bee.set_zorder(1)
        ax_bee.patch.set_alpha(0)
        colors = cmap1(np.linspace(0, 1, len(y_coords)))
        
        ax_bar.barh(
            y=y_coords,
            width=bar_widths,
            height=0.6,
            alpha=0.3,
            color=colors,
            edgecolor="none",
            zorder=0
        )
        
        bar_xlim = max(bar_widths) * 1.1 if max(bar_widths) > 0 else 1.0
        ax_bar.set_xlim(0, bar_xlim)
        ax_bar.set_xticks(np.linspace(0, bar_xlim, 7))
        ax_bar.set_xticklabels([f"{x:.2f}" for x in np.linspace(0, bar_xlim, 7)], 
                              fontsize=11, color='black')
        ax_bar.set_xlabel("Mean(|SHAP| value)", fontsize=11, color='black')
        ax_bar.set_yticks([])
        
        shap_xlim = round(abs(shap_vals_sorted).max() * 2, 1)
        ax_bee.set_xlim(-shap_xlim, shap_xlim)
        ax_bee.set_xticks(np.linspace(-shap_xlim, shap_xlim, 7))
        ax_bee.set_xlabel("SHAP value (impact on model output)", fontsize=11, color='black')
        
        ax_bee.set_yticks(y_coords)
        ax_bee.set_yticklabels(y_order, fontsize=11, color='black')
        ax_bar.set_ylim(ax_bee.get_ylim())
        
        for ax in [ax_bee, ax_bar]:
            ax.tick_params(axis='both', which='major', length=4, width=1, colors='black', labelsize=11)
        
        for y, p in zip(y_coords, bar_pcts):
            ax_bar.text(0.01 * bar_xlim, y, f"({p:.1f}%)", va="center", ha="left", 
                       fontsize=10, color="black", usetex=False)
        
        plt.savefig(f"{city_name}_shap_beeswarm.pdf", 
                   format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(f"{city_name}_shap_beeswarm.png", 
                   format='png', bbox_inches='tight', dpi=300)
        plt.close()
        print("✓ SHAP beeswarm plot saved")
        
    except Exception as e:
        print(f"Error creating beeswarm plot: {str(e)}")

def create_interaction_plots_original(shap_interaction_values, X_test, city_name):
    try:
        from matplotlib.colors import LinearSegmentedColormap
        import shap
        
        feature_names = X_test.columns.tolist()
        n_features = len(feature_names)
        
        print(f"Creating interaction plot ({n_features} rows × {n_features} columns)...")
        
        subplot_width = 4.5 
        subplot_height = 2.8
        fig_width = n_features * subplot_width
        fig_height = n_features * subplot_height
        
        fig, axes = plt.subplots(n_features, n_features, figsize=(fig_width, fig_height), dpi=300)
        fig.patch.set_facecolor('white')
        
        label_fontsize = 20
        tick_fontsize = 18
        title_fontsize = 24
        diag_fontsize = 22
        colorbar_fontsize = 20 
        
        colors = [
            (0.25, 0.7, 1.0),
            (1.0, 0.35, 0.25)
        ]
        custom_cmap = LinearSegmentedColormap.from_list('light_blue_red', colors)
        
        shap.plots._utils.colors.red_blue = custom_cmap
        
        for row_idx, main_feature in enumerate(feature_names):
            for col_idx, other_feature in enumerate(feature_names):
                ax = axes[row_idx, col_idx]
                
                if main_feature == other_feature:
                    ax.text(0.5, 0.5, f"{main_feature}", 
                           transform=ax.transAxes, ha='center', va='center', 
                           fontsize=diag_fontsize, fontweight='bold')
                    ax.set_xlabel('')
                    ax.set_ylabel('')
                    ax.set_xticks([])
                    ax.set_yticks([])
                    continue
                
                try:
                    shap.dependence_plot(
                        (other_feature, main_feature),
                        shap_interaction_values,
                        X_test,
                        display_features=X_test,
                        show=False,
                        axis_color='black',
                        ax=ax,
                        cmap=custom_cmap
                    )
                    
                    for collection in ax.collections:
                        if hasattr(collection, "_sizes"):
                            collection._sizes = [25] * len(collection._sizes)
                    
                    ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
                    
                    for spine in ax.spines.values():
                        spine.set_color('black')
                        spine.set_linewidth(1.2)
                    
                    ax.tick_params(colors='black', which='both', width=1.2, 
                                  labelsize=tick_fontsize, pad=4, length=6)
                    
                    if row_idx == n_features - 1:
                        ax.set_xlabel(f"{other_feature}", fontsize=label_fontsize, color='black')
                    else:
                        ax.set_xlabel('')
                    
                    if col_idx == 0:
                        ax.set_ylabel("Interaction value", fontsize=label_fontsize, color='black', labelpad=5)
                    else:
                        ax.set_ylabel('')
                    
                    ax.spines['right'].set_visible(True)
                    ax.spines['top'].set_visible(True)
                    
                except Exception as e:
                    print(f"Error plotting {main_feature} vs {other_feature}: {str(e)}")
                    ax.clear()
                    ax.text(0.5, 0.5, f"Error", transform=ax.transAxes, 
                           ha='center', va='center', fontsize=label_fontsize)
        
        for row_idx in range(n_features):
            for col_idx in range(n_features):
                if row_idx != col_idx:
                    ax = axes[row_idx, col_idx]
                    main_feature = feature_names[row_idx]
                    
                    for cbar in fig.axes:
                        if hasattr(cbar, 'get_label') and cbar.get_label() == main_feature:
                            pos = cbar.ax.get_position()
                            new_pos = [pos.x0 - 0.02, pos.y0, 0.03, pos.height]
                            cbar.ax.set_position(new_pos)
                            
                            cbar.locator = MaxNLocator(nbins=3)
                            cbar.update_ticks()
                            
                            cbar.ax.tick_params(labelsize=colorbar_fontsize)
                            cbar.set_label(main_feature, fontsize=colorbar_fontsize, labelpad=10)
        
        fig.suptitle(f"{city_name} - SHAP Interaction Effects", 
                    fontsize=title_fontsize, y=0.98)
        
        plt.subplots_adjust(left=0.1, right=0.92, bottom=0.08, top=0.95, wspace=0.45, hspace=0.25)
        
        filename = f"{city_name}_interaction_effects.pdf"
        plt.savefig(filename, format='pdf', bbox_inches='tight', dpi=300)
        plt.savefig(filename.replace('.pdf', '.png'), format='png', bbox_inches='tight', dpi=300)
        plt.close()
        
        print(f"✓ interaction matrix saved: {filename}")
        
    except Exception as e:
        print(f"Error creating interaction plot: {str(e)}")
        import traceback
        traceback.print_exc()

# ======================
# GAM Visualization Functions
# ======================

def create_gam_plots_complete(shap_values_Explanation, X_test, city_name):
    try:
        plt.style.use('default')
        cm_to_inch = 1 / 2.54
        
        axis_box_width_cm = 6
        axis_box_height_cm = 6
        n_rows = 3
        n_cols = 4
        padding_factor = 1.2
        
        subplot_width_cm = axis_box_width_cm * padding_factor
        subplot_height_cm = axis_box_height_cm * padding_factor
        total_width_cm = n_cols * subplot_width_cm
        total_height_cm = n_rows * subplot_height_cm
        
        fig_width = total_width_cm * cm_to_inch
        fig_height = total_height_cm * cm_to_inch
        
        base_fontsize = 14
        
        rcParams.update({
            'font.family': 'Times New Roman',
            'font.size': base_fontsize,
            'axes.labelsize': base_fontsize,
            'axes.titlesize': base_fontsize + 2,
            'xtick.labelsize': base_fontsize - 1,
            'ytick.labelsize': base_fontsize - 1,
            'legend.fontsize': base_fontsize - 2,
            'lines.linewidth': 1.5,
            'grid.linewidth': 0.6,
            'lines.markersize': 8,
            'savefig.dpi': 600,
            'savefig.format': 'pdf',
            'axes.edgecolor': 'black',
            'axes.labelcolor': 'black',
            'xtick.color': 'black',
            'ytick.color': 'black',
            'xtick.major.size': 4.0,
            'xtick.major.width': 1.0,
            'ytick.major.size': 4.0,
            'ytick.major.width': 1.0,
            'axes.spines.top': True,
            'axes.spines.right': True,
            'axes.spines.bottom': True,
            'axes.spines.left': True,
            'figure.titlesize': base_fontsize + 2,
            'pdf.fonttype': 42,
            'ps.fonttype': 42,
            'svg.fonttype': 'none',
            'text.usetex': False,
        })
        
        COLOR_PALETTE = {
            'main': '#3b5b92',
            'ci': '#8395b1',
            'positive': '#36a168',
            'negative': '#e05263',
            'data': '#e0e0e0',
            'zero_line': '#000000',
            'tipping_point': '#f0746e',
            'background': '#f9f9f9'
        }
        
        mean_abs_shap = np.mean(np.abs(shap_values_Explanation.values), axis=0)
        feature_importance = pd.DataFrame({
            'feature': X_test.columns,
            'importance': mean_abs_shap
        }).sort_values('importance', ascending=False)
        
        selected_features = feature_importance['feature'].tolist()
        
        print("All features sorted by importance:")
        for i, feature in enumerate(selected_features, 1):
            importance_value = feature_importance[feature_importance['feature'] == feature]['importance'].values[0]
            print(f"{i}. {feature}: {importance_value:.4f}")
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height))
        fig.patch.set_facecolor('white')
        
        if not isinstance(axes, np.ndarray):
            axes = np.array([[axes]])
        elif axes.ndim == 1:
            axes = axes.reshape(1, -1)
        
        axes_flat = axes.flatten()
        
        plt.subplots_adjust(
            left=0.07, right=0.98, bottom=0.07, top=0.92, wspace=0.3, hspace=0.4
        )
        
        for i, feature_name in enumerate(selected_features):
            if i >= len(axes_flat):
                break
                
            if feature_name not in X_test.columns:
                continue
                
            n = list(X_test.columns).index(feature_name)
            X_feature = X_test[feature_name].values.reshape(-1, 1)
            y_shap = shap_values_Explanation.values[:, n].reshape(-1, 1)
            
            gam = LinearGAM(s(0, n_splines=20, spline_order=3, lam=3)).gridsearch(X_feature, y_shap)
            R_squared = gam.statistics_['pseudo_r2']['explained_deviance']
            p_value = gam.statistics_['p_values'][0]
            
            XX = gam.generate_X_grid(term=0, n=300)
            y_pred = gam.predict(XX)
            ci = gam.prediction_intervals(XX, width=0.95)
            
            zero_crossings = np.where(np.diff(np.sign(y_pred)))[0]
            tipping_points = [(XX[i][0], y_pred[i]) for i in zero_crossings if i < len(XX)]
            
            ax = axes_flat[i]
            show_ylabel = (i % n_cols == 0)
            
            plot_scientific_gam_style_fixed(ax, XX, y_pred, ci, feature_name, tipping_points, 
                                          p_value, show_ylabel, COLOR_PALETTE, base_fontsize)
            
            ax.set_title(city_name, fontsize=base_fontsize, color='black', pad=6)
            ax.text(0.05, 0.90, f"R² = {R_squared:.2f}",
                    transform=ax.transAxes, ha='left', va='top',
                    fontsize=base_fontsize - 1, color='black', usetex=False)
            
            data_min = min(ci[:, 0].min(), y_pred.min())
            data_max = max(ci[:, 1].max(), y_pred.max())
            data_range = data_max - data_min
            
            if data_range < 1.0:
                padding_factor_local = 0.2
            elif data_range < 2.0:
                padding_factor_local = 0.15
            else:
                padding_factor_local = 0.1
            
            y_padding = padding_factor_local * data_range
            y_min = data_min - y_padding
            y_max = data_max + y_padding
            
            if y_min > 0 and data_min < 0.1 * data_range:
                y_min = -0.05 * data_range
            if y_max < 0 and data_max > -0.1 * data_range:
                y_max = 0.05 * data_range
            
            ax.set_ylim([y_min, y_max])
            ax.tick_params(axis='x', labelsize=base_fontsize-1)
            ax.tick_params(axis='y', labelsize=base_fontsize-1)
            
            xlabel = ax.get_xlabel()
            ax.set_xlabel(xlabel, fontsize=base_fontsize, color='black', labelpad=2)
            ax.tick_params(axis='x', which='major', pad=3)
            ax.tick_params(axis='y', which='major', pad=3)
        
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)
        
        plt.savefig(f"{city_name}_gam_main_effects.pdf", 
                   format='pdf', dpi=600, bbox_inches='tight',
                   facecolor=fig.get_facecolor(), edgecolor='none')
        plt.savefig(f"{city_name}_gam_main_effects.png", 
                   format='png', dpi=600, bbox_inches='tight',
                   facecolor=fig.get_facecolor(), edgecolor='none')
        plt.close()
        print("✓ GAM main effects plot saved")
        
    except Exception as e:
        print(f"Error creating GAM plot: {str(e)}")

def plot_scientific_gam_style_fixed(ax, XX, y_pred, ci, feature_name, tipping_points, 
                                    p_value, show_ylabel, COLOR_PALETTE, base_fontsize=14):

    ax.set_facecolor(COLOR_PALETTE['background'])
    
    ax.axhline(0, color=COLOR_PALETTE['zero_line'],
               linestyle=(0, (5, 2)), linewidth=1.0, alpha=0.9, zorder=2)
    
    ax.fill_between(XX.flatten(), ci[:, 0], ci[:, 1],
                    color=COLOR_PALETTE['ci'], alpha=0.4, zorder=3, edgecolor='none')
    
    ax.plot(XX, y_pred, color=COLOR_PALETTE['main'], linewidth=1.8, zorder=5)
    
    if tipping_points:
        for i, (x, y) in enumerate(tipping_points):
            ax.axvline(x, color=COLOR_PALETTE['tipping_point'],
                       linestyle='--', linewidth=1.0, alpha=0.8)
            ax.scatter(x, y, color=COLOR_PALETTE['tipping_point'],
                       s=60, zorder=6, edgecolor='white', linewidth=0.8)
            
            xlim = ax.get_xlim()
            ylim = ax.get_ylim()
            x_range = xlim[1] - xlim[0]
            y_range = ylim[1] - ylim[0]
            
            x_offset = x_range * 0.05
            y_offset = y_range * 0.05
            
            if x > (xlim[0] + xlim[1]) / 2:
                text_x = x - x_offset
                ha = 'right'
            else:
                text_x = x + x_offset
                ha = 'left'
            
            if y > (ylim[0] + ylim[1]) / 2:
                text_y = y - y_offset
                va = 'top'
            else:
                text_y = y + y_offset
                va = 'bottom'
            
            text_x = max(xlim[0] + x_range * 0.02, min(text_x, xlim[1] - x_range * 0.02))
            text_y = max(ylim[0] + y_range * 0.02, min(text_y, ylim[1] - y_range * 0.02))
            
            ax.text(text_x, text_y, f'{x:.2f}',
                    fontsize=base_fontsize - 2, color='black', zorder=10,
                    ha=ha, va=va)
    
    pos_mask = y_pred > 0
    if any(pos_mask):
        ax.fill_between(XX.flatten(), 0, y_pred, where=pos_mask,
                        color=COLOR_PALETTE['positive'], alpha=0.15, zorder=0)
    
    neg_mask = y_pred <= 0
    if any(neg_mask):
        ax.fill_between(XX.flatten(), 0, y_pred, where=neg_mask,
                        color=COLOR_PALETTE['negative'], alpha=0.15, zorder=0)
    
    ax.tick_params(axis='both', which='major', length=4.0, width=1.0, colors='black', pad=4)
    ax.tick_params(axis='both', which='minor', size=0)
    
    ax.set_xlabel(feature_name, labelpad=3, fontsize=base_fontsize, color='black')
    
    if show_ylabel:
        ax.set_ylabel('', labelpad=3, fontsize=base_fontsize, color='black')
    
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color('black')
    
    if p_value is not None:
        if p_value < 0.001:
            p_text = "p < 0.001"
        elif p_value < 0.01:
            p_text = "p < 0.01"
        else:
            p_text = f"p = {p_value:.3f}"
        
        ax.text(0.90, 0.90, p_text, transform=ax.transAxes, ha='right', va='top',
                fontsize=base_fontsize - 1, color='black')
    
    for text in ax.texts:
        text.set_color('black')
    
    for label in ax.get_xticklabels():
        label.set_color('black')
        label.set_fontsize(base_fontsize - 1)
    for label in ax.get_yticklabels():
        label.set_color('black')
        label.set_fontsize(base_fontsize - 1)

# ======================
# Export Functions
# ======================

def generate_r2_summary_table(base_path, city_name):
    if not all_cities_r2_data:
        print("No R2 data to summarize")
        return
    
    r2_df = pd.DataFrame(all_cities_r2_data)
    r2_df = r2_df.sort_values('Test_R2', ascending=False)
    r2_df['R2_Rank'] = range(1, len(r2_df) + 1)
    
    columns_order = ['R2_Rank', 'City', 'Train_R2', 'Test_R2', 'Train_RMSE', 'Test_RMSE', 'Sample_Size', 'Num_Features']
    r2_df = r2_df[columns_order]
    
    output_path = os.path.join(base_path, f"{city_name}_model_evaluation_metrics.csv")
    r2_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ {city_name}_model_evaluation_metrics.csv saved: {output_path}")

def generate_best_params_summary_table(base_path, city_name):
    if not all_cities_best_params:
        print("No best parameters data to summarize")
        return
    
    params_df = pd.DataFrame(all_cities_best_params)
    cols = ['City'] + [col for col in params_df.columns if col != 'City']
    params_df = params_df[cols]
    params_df = params_df.sort_values('City')
    
    output_path = os.path.join(base_path, f"{city_name}_best_params_summary.csv")
    params_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ {city_name}_best_params_summary.csv saved: {output_path}")

def generate_complete_features_summary_table(base_path, city_name):
    if not all_cities_complete_features:
        print("No complete feature data to summarize")
        return
    
    complete_df = pd.DataFrame(all_cities_complete_features)
    output_path = os.path.join(base_path, f"{city_name}_feature_importance_summary.csv")
    complete_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✓ {city_name}_feature_importance_summary.csv saved: {output_path}")

# ======================
# Program Entry Point
# ======================
if __name__ == "__main__":

    csv_file = INPUT_CSV_FILE
    
    if not os.path.exists(csv_file):
        print(f"Error: File {csv_file} does not exist!")
    else:
        output_dir = os.path.dirname(csv_file)
        city_name = os.path.splitext(os.path.basename(csv_file))[0]
        
        print(f"Processing single file: {csv_file}")
        print(f"Output directory: {output_dir}")
        print(f"City name: {city_name}")
        print(f"Features used: {SPECIFIED_FEATURES}")
        
        # Reset global variables for new run
        all_cities_r2_data = []
        all_cities_best_params = []
        all_cities_complete_features = []
        
        # Execute main processing function
        result = process_single_city_complete(csv_file, city_name, output_dir)
        
        # Report results
        if result and result['Status'] == 'Success':
            print(f"\n✓ Processing completed. Results saved to: {output_dir}")
        else:
            print(f"\n✗ Processing failed: {result['Status'] if result else 'Unknown error'}")